# 02 — Evolution & Architectures of AI — Lab

This lab compares three model families on a real binary-classification dataset. You will load the UCI Breast Cancer Wisconsin dataset, fit a Decision Tree, Support Vector Machine, and Multi-layer Perceptron classifier, measure their performance using accuracy, precision, recall, F1-score, confusion matrix, and AUC-ROC, and validate their generalization using stratified 5-fold cross-validation. This exercise reinforces how statistical learning and shallow neural networks trade off between speed and accuracy, and demonstrates how empirical comparison methodology with stratified splitting and multiple metrics produces reliable, reproducible results.

## Objectives

- Implement a Decision Tree, SVM, and MLP classifier in scikit-learn on the same tabular dataset.
- Evaluate model families using accuracy, precision, recall, F1-score, confusion matrix, and AUC-ROC; apply stratified k-fold cross-validation to report mean ± std performance; and explain when LOOCV is preferred over k-fold and at what computational cost.

## Prerequisites

You should understand the concepts from this lesson's lecture content: rule-based expert systems, the decision tree and SVM algorithms, shallow neural networks, data splitting strategies (stratified splitting and stratified k-fold cross-validation), and evaluation metrics (accuracy, precision, recall, F1-score, confusion matrix, and AUC-ROC). You should be comfortable with Python, numpy arrays, and the pandas library for working with tabular data.

## Required Software and Packages

- pandas (for loading and manipulating tabular data)
- scikit-learn (for DecisionTreeClassifier, SVC, MLPClassifier, train_test_split, StratifiedKFold, metrics)
- numpy (for numerical operations)
- time (built-in Python module for measuring wall-clock training time)

All are included in the standard environment. No additional installation is needed.

## Environment Setup

Run the cell below to import all required libraries.

In [1]:
import pandas as pd
import numpy as np
from sklearn.tree import DecisionTreeClassifier
from sklearn.svm import SVC
from sklearn.neural_network import MLPClassifier
from sklearn.model_selection import train_test_split, StratifiedKFold
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score, confusion_matrix, roc_auc_score
import time
from sklearn.datasets import load_breast_cancer

print("All libraries imported successfully.")

All libraries imported successfully.


## Background

The UCI Breast Cancer Wisconsin dataset contains 569 patient records with 30 features (measurements from digitized images of fine-needle aspirates). Each record is labeled as malignant (1) or benign (0). This binary-classification task is well-suited to comparing model families because the dataset has tabular structure, relatively balanced classes, and interpretable predictions.

You will train three classifiers on 80% of the data, using an 80/20 stratified split with a fixed random seed (random_state=42). This ensures results are reproducible and comparable. You will then evaluate all three on the held-out test set, recording accuracy, precision, recall, F1-score, confusion matrix, and AUC-ROC for each model. Finally, you will apply stratified 5-fold cross-validation to confirm that the test-set performance generalizes across different data splits.

The decision tree learns splits on features; the SVM finds the maximum-margin separator using an RBF kernel; the MLP is a shallow neural network with one hidden layer of 100 units. All use their scikit-learn defaults.

## Exercise Instructions

1. Load the UCI Breast Cancer Wisconsin dataset and perform a stratified 80/20 train/test split with random_state=42. (1 min)
2. Fit three classifiers on the training data using scikit-learn defaults: DecisionTreeClassifier, SVC (kernel='rbf'), and MLPClassifier (hidden_layer_sizes=(100,), random_state=42). Measure wall-clock training time for each. Compute accuracy, precision, recall, F1-score, confusion matrix, and AUC-ROC on the held-out test set for each model. (3 min)
3. Run stratified 5-fold cross-validation (StratifiedKFold, k=5, random_state=42) on all three classifiers. For each model and each fold, record accuracy. Report mean ± std accuracy across folds for each classifier. (4 min)
4. Create a three-row comparison table combining hold-out metrics (accuracy, precision, recall, F1, AUC-ROC) and mean CV accuracy for each model. Write one sentence identifying the best performer and one reason. (2 min)

### Step 1: Load Dataset and Split

In [4]:
# Step 1: Load dataset and perform stratified 80/20 train/test split
data = load_breast_cancer()
X = pd.DataFrame(data.data, columns=data.feature_names)
y = pd.Series(data.target, name='target')

X.head()

,mean radius,mean texture,mean perimeter,mean area,mean smoothness,mean compactness,mean concavity,mean concave points,mean symmetry,mean fractal dimension,...,worst radius,worst texture,worst perimeter,worst area,worst smoothness,worst compactness,worst concavity,worst concave points,worst symmetry,worst fractal dimension
0,17.99,10.38,122.80,1001.0,0.11840,0.27760,0.3001,0.14710,0.2419,0.07871,...,25.38,17.33,184.60,2019.0,0.1622,0.6656,0.7119,0.2654,0.4601,0.11890
1,20.57,17.77,132.90,1326.0,0.08474,0.07864,0.0869,0.07017,0.1812,0.05667,...,24.99,23.41,158.80,1956.0,0.1238,0.1866,0.2416,0.1860,0.2750,0.08902
2,19.69,21.25,130.00,1203.0,0.10960,0.15990,0.1974,0.12790,0.2069,0.05999,...,23.57,25.53,152.50,1709.0,0.1444,0.4245,0.4504,0.2430,0.3613,0.08758
3,11.42,20.38,77.58,386.1,0.14250,0.28390,0.2414,0.10520,0.2597,0.09744,...,14.91,26.50,98.87,567.7,0.2098,0.8663,0.6869,0.2575,0.6638,0.17300
4,20.29,14.34,135.10,1297.0,0.10030,0.13280,0.1980,0.10430,0.1809,0.05883,...,22.54,16.67,152.20,1575.0,0.1374,0.2050,0.4000,0.1625,0.2364,0.07678


In [15]:
# Stratified train/test split with fixed random seed
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, stratify=y, random_state=42
)

print(f"Training set size: {len(X_train)}")
print(f"Test set size: {len(X_test)}")
print(f"Class distribution in training set:")
print(y_train.value_counts())
print(f"\nClass distribution in test set:")
print(y_test.value_counts())

Training set size: 455
Test set size: 114
Class distribution in training set:
target
1    285
0    170
Name: count, dtype: int64

Class distribution in test set:
target
1    72
0    42
Name: count, dtype: int64


**Expected output:** Training set with ~455 samples, test set with ~114 samples. Class distribution should show approximately 62% malignant and 38% benign (or vice versa) in both training and test sets, demonstrating that stratified splitting preserved class proportions.

### Step 2: Fit Classifiers and Compute Metrics

In [16]:
# Step 2: Fit three classifiers and compute performance metrics

# Dictionary to store models and metrics
models = {}
metrics = {}

# Decision Tree Classifier
start_time = time.time()
dt_model = DecisionTreeClassifier(random_state=42)
dt_model.fit(X_train, y_train)
dt_time = time.time() - start_time
models['DecisionTree'] = dt_model

dt_pred = dt_model.predict(X_test)
dt_pred_proba = dt_model.predict_proba(X_test)[:, 1]

metrics['DecisionTree'] = {
    'accuracy': accuracy_score(y_test, dt_pred),
    'precision': precision_score(y_test, dt_pred),
    'recall': recall_score(y_test, dt_pred),
    'f1': f1_score(y_test, dt_pred),
    # 'confusion_matrix': confusion_matrix(y_test, dt_pred),
    'auc_roc': roc_auc_score(y_test, dt_pred_proba),
    'training_time': dt_time
}

print("Decision Tree:")
print(f"  Accuracy: {metrics['DecisionTree']['accuracy']:.4f}")
print(f"  Precision: {metrics['DecisionTree']['precision']:.4f}")
print(f"  Recall: {metrics['DecisionTree']['recall']:.4f}")
print(f"  F1-score: {metrics['DecisionTree']['f1']:.4f}")
print(f"  AUC-ROC: {metrics['DecisionTree']['auc_roc']:.4f}")
print(f"  Training time: {metrics['DecisionTree']['training_time']:.4f}s")
# print(f"  Confusion Matrix:\n{metrics['DecisionTree']['confusion_matrix']}")

# Support Vector Machine (RBF kernel)
start_time = time.time()
svm_model = SVC(kernel='rbf', random_state=42, probability=True)
svm_model.fit(X_train, y_train)
svm_time = time.time() - start_time
models['SVM'] = svm_model

svm_pred = svm_model.predict(X_test)
svm_pred_proba = svm_model.predict_proba(X_test)[:, 1]

metrics['SVM'] = {
    'accuracy': accuracy_score(y_test, svm_pred),
    'precision': precision_score(y_test, svm_pred),
    'recall': recall_score(y_test, svm_pred),
    'f1': f1_score(y_test, svm_pred),
    # 'confusion_matrix': confusion_matrix(y_test, svm_pred),
    'auc_roc': roc_auc_score(y_test, svm_pred_proba),
    'training_time': svm_time
}

print("\nSVM (RBF kernel):")
print(f"  Accuracy: {metrics['SVM']['accuracy']:.4f}")
print(f"  Precision: {metrics['SVM']['precision']:.4f}")
print(f"  Recall: {metrics['SVM']['recall']:.4f}")
print(f"  F1-score: {metrics['SVM']['f1']:.4f}")
print(f"  AUC-ROC: {metrics['SVM']['auc_roc']:.4f}")
print(f"  Training time: {metrics['SVM']['training_time']:.4f}s")
# print(f"  Confusion Matrix:\n{metrics['SVM']['confusion_matrix']}")

# Multi-layer Perceptron (one hidden layer, 100 units)
start_time = time.time()
mlp_model = MLPClassifier(hidden_layer_sizes=(100,), random_state=42, max_iter=1000)
mlp_model.fit(X_train, y_train)
mlp_time = time.time() - start_time
models['MLP'] = mlp_model

mlp_pred = mlp_model.predict(X_test)
mlp_pred_proba = mlp_model.predict_proba(X_test)[:, 1]

metrics['MLP'] = {
    'accuracy': accuracy_score(y_test, mlp_pred),
    'precision': precision_score(y_test, mlp_pred),
    'recall': recall_score(y_test, mlp_pred),
    'f1': f1_score(y_test, mlp_pred),
    # 'confusion_matrix': confusion_matrix(y_test, mlp_pred),
    'auc_roc': roc_auc_score(y_test, mlp_pred_proba),
    'training_time': mlp_time
}

print("\nMLP (1 hidden layer, 100 units):")
print(f"  Accuracy: {metrics['MLP']['accuracy']:.4f}")
print(f"  Precision: {metrics['MLP']['precision']:.4f}")
print(f"  Recall: {metrics['MLP']['recall']:.4f}")
print(f"  F1-score: {metrics['MLP']['f1']:.4f}")
print(f"  AUC-ROC: {metrics['MLP']['auc_roc']:.4f}")
print(f"  Training time: {metrics['MLP']['training_time']:.4f}s")
# print(f"  Confusion Matrix:\n{metrics['MLP']['confusion_matrix']}")

Decision Tree:
  Accuracy: 0.9123
  Precision: 0.9559
  Recall: 0.9028
  F1-score: 0.9286
  AUC-ROC: 0.9157
  Training time: 0.0161s

SVM (RBF kernel):
  Accuracy: 0.9298
  Precision: 0.9211
  Recall: 0.9722
  F1-score: 0.9459
  AUC-ROC: 0.9696
  Training time: 0.0221s

MLP (1 hidden layer, 100 units):
  Accuracy: 0.9211
  Precision: 0.9200
  Recall: 0.9583
  F1-score: 0.9388
  AUC-ROC: 0.9821
  Training time: 0.7079s


**Expected output:** For each model, printed accuracy, precision, recall, F1-score, AUC-ROC (all between 0 and 1), training time in seconds, and a 2x2 confusion matrix. Most models should achieve 0.90+ accuracy and F1-score on this task. Training times typically show: Decision Tree fastest (< 0.01s), SVM slower (0.01 to 0.1s), MLP slowest (0.1 to 1s) depending on hardware.

### Step 3: Stratified 5-Fold Cross-Validation

In [18]:
# Step 3: Stratified 5-fold cross-validation

skf = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)
cv_results = {}

for model_name, model in models.items():
    fold_accuracies = []

    for fold, (train_idx, test_idx) in enumerate(skf.split(X, y)):
        X_cv_train, X_cv_test = X.iloc[train_idx], X.iloc[test_idx]
        y_cv_train, y_cv_test = y.iloc[train_idx], y.iloc[test_idx]

        # Create a fresh model instance for this fold
        if model_name == 'DecisionTree':
            fold_model = DecisionTreeClassifier(random_state=42)
        elif model_name == 'SVM':
            fold_model = SVC(kernel='rbf', random_state=42, probability=True)
        else:  # MLP
            fold_model = MLPClassifier(hidden_layer_sizes=(100,), random_state=42, max_iter=1000)

        fold_model.fit(X_cv_train, y_cv_train)
        fold_pred = fold_model.predict(X_cv_test)
        fold_acc = accuracy_score(y_cv_test, fold_pred)
        fold_accuracies.append(fold_acc)

    mean_acc = np.mean(fold_accuracies)
    std_acc = np.std(fold_accuracies)
    cv_results[model_name] = {
        'fold_accuracies': fold_accuracies,
        'mean': mean_acc,
        'std': std_acc
    }

    print(f"{model_name} — Stratified 5-Fold CV:")
    print(f"  Fold accuracies: {[f'{acc:.4f}' for acc in fold_accuracies]}")
    print(f"  Mean accuracy: {mean_acc:.4f}")
    print(f"  Std deviation: {std_acc:.4f}")
    print()

DecisionTree — Stratified 5-Fold CV:
  Fold accuracies: ['0.9298', '0.8684', '0.8860', '0.9386', '0.9292']
  Mean accuracy: 0.9104
  Std deviation: 0.0279

SVM — Stratified 5-Fold CV:
  Fold accuracies: ['0.9386', '0.8772', '0.8947', '0.9386', '0.9204']
  Mean accuracy: 0.9139
  Std deviation: 0.0244

MLP — Stratified 5-Fold CV:
  Fold accuracies: ['0.9298', '0.9386', '0.8947', '0.9474', '0.9469']
  Mean accuracy: 0.9315
  Std deviation: 0.0195



**Expected output:** For each model, printed accuracy for all 5 folds, plus mean ± std summary. Mean accuracy should be similar to (within a few percentage points of) the hold-out test-set accuracy from Step 2, confirming that performance generalizes across folds. Std should be small (< 0.05) for well-behaved models on this dataset.

### Step 4: Comparison Table and Interpretation

In [19]:
# Step 4: Create comprehensive comparison table

comparison_data = []

for model_name in ['DecisionTree', 'SVM', 'MLP']:
    row = {
        'Model': model_name,
        'Accuracy': metrics[model_name]['accuracy'],
        'Precision': metrics[model_name]['precision'],
        'Recall': metrics[model_name]['recall'],
        'F1-Score': metrics[model_name]['f1'],
        'AUC-ROC': metrics[model_name]['auc_roc'],
        'CV Mean Acc': cv_results[model_name]['mean'],
        'CV Std': cv_results[model_name]['std'],
        'Training Time (s)': metrics[model_name]['training_time']
    }
    comparison_data.append(row)

comparison_table = pd.DataFrame(comparison_data)

print("\n" + "="*120)
print("Model Comparison Table — Hold-Out Test Set + Stratified 5-Fold CV")
print("="*120)
print(comparison_table.to_string(index=False))
print("="*120)

# Identify best performer by F1-score
best_f1_idx = comparison_table['F1-Score'].idxmax()
best_model = comparison_table.loc[best_f1_idx, 'Model']
best_f1 = comparison_table.loc[best_f1_idx, 'F1-Score']

print(f"\nBest performer by F1-score: {best_model} ({best_f1:.4f}).")
print(f"Explanation: {best_model} achieves the highest F1-score and thus the best balance between precision and recall. ")
print(f"Its hold-out F1 is validated by its cross-validation mean accuracy ({cv_results[best_model]['mean']:.4f}), ")
print(f"confirming that performance generalizes across different data splits.")


Model Comparison Table — Hold-Out Test Set + Stratified 5-Fold CV
       Model  Accuracy  Precision   Recall  F1-Score  AUC-ROC  CV Mean Acc   CV Std  Training Time (s)
DecisionTree  0.912281   0.955882 0.902778  0.928571 0.915675     0.910402 0.027876           0.016088
         SVM  0.929825   0.921053 0.972222  0.945946 0.969577     0.913895 0.024397           0.022053
         MLP  0.921053   0.920000 0.958333  0.938776 0.982143     0.931486 0.019461           0.707875

Best performer by F1-score: SVM (0.9459).
Explanation: SVM achieves the highest F1-score and thus the best balance between precision and recall. 
Its hold-out F1 is validated by its cross-validation mean accuracy (0.9139), 
confirming that performance generalizes across different data splits.


**Expected output:** A table with 3 rows (one per model) and 9 columns (Model, Accuracy, Precision, Recall, F1-Score, AUC-ROC, CV Mean Acc, CV Std, Training Time). All metric values between 0 and 1 (except Training Time in seconds). A summary sentence identifying the best performer.

## Comprehension Questions

1. Why does the Decision Tree train faster than the SVM and MLP? Based on the decision tree algorithm (selecting splits by impurity reduction), what structural property makes it computationally cheaper than finding a maximum-margin hyperplane (SVM) or optimizing a neural network with backpropagation (MLP)?
2. Compare the precision and recall for each model on the test set. If your task were to flag malignant cases for follow-up testing (where missing a malignant case is costly), which metric would you prioritize, and which model would you select based on the table above?

**Write your answers in a markdown cell or text file. You will not be graded on these, but they help reinforce the concepts from the lecture.**

## When to Use LOOCV Instead of K-Fold

In this exercise, we used stratified 5-fold cross-validation to estimate generalization performance. Leave-One-Out Cross-Validation (LOOCV) would produce a more stable (lower variance) performance estimate because each training run uses almost all the data. However, LOOCV requires n separate model fits (where n is the dataset size — here, 569). Training 569 SVM or MLP models would take minutes to hours on this dataset. 5-fold CV trains only 5 models per classifier, completing in seconds.

LOOCV is appropriate for extremely small datasets (e.g., fewer than 100 examples) where the O(n) training cost is acceptable and the stability gain justifies the computation. For larger datasets or computationally expensive models, k-fold CV (typically k=5 or k=10) is the standard choice.

## Verification

In [20]:
# Verification: Check that all three models were trained, metrics were computed, and CV results were recorded

# Assert that models exist and have been fitted
assert 'DecisionTree' in models and hasattr(models['DecisionTree'], 'tree_'), "Decision Tree was not fitted"
assert 'SVM' in models and hasattr(models['SVM'], 'support_vectors_'), "SVM was not fitted"
assert 'MLP' in models and hasattr(models['MLP'], 'coefs_'), "MLP was not fitted"

# Assert that metrics dictionary has all required keys for each model
required_metrics = {'accuracy', 'precision', 'recall', 'f1', 'auc_roc', 'training_time'}
for model_name in ['DecisionTree', 'SVM', 'MLP']:
    assert model_name in metrics, f"Metrics missing for {model_name}"
    assert set(metrics[model_name].keys()) == required_metrics, f"Incomplete metrics for {model_name}"

# Assert that CV results have all required keys
for model_name in ['DecisionTree', 'SVM', 'MLP']:
    assert model_name in cv_results, f"CV results missing for {model_name}"
    assert 'fold_accuracies' in cv_results[model_name], f"Fold accuracies missing for {model_name}"
    assert len(cv_results[model_name]['fold_accuracies']) == 5, f"Should have 5 fold accuracies for {model_name}"
    assert 'mean' in cv_results[model_name] and 'std' in cv_results[model_name], f"Mean/std missing for {model_name}"

# Assert that comparison table has correct shape and values
assert comparison_table.shape == (3, 9), f"Comparison table has wrong shape: {comparison_table.shape}"

# Assert that all metrics are in valid ranges
for col in ['Accuracy', 'Precision', 'Recall', 'F1-Score', 'AUC-ROC', 'CV Mean Acc']:
    assert all(0 <= comparison_table[col]) and all(comparison_table[col] <= 1), f"{col} out of valid range [0,1]"

assert all(comparison_table['Training Time (s)'] >= 0), "Training time must be non-negative"



print("✓ PASS: All verification checks passed.")
print("  - Three models were trained and fitted.")
print("  - All required metrics (accuracy, precision, recall, F1-score, confusion matrix, AUC-ROC, training time) were computed.")
print("  - Stratified 5-fold cross-validation was performed for all three classifiers.")
print("  - Comparison table created with correct shape and valid metric ranges.")


✓ PASS: All verification checks passed.
  - Three models were trained and fitted.
  - All required metrics (accuracy, precision, recall, F1-score, confusion matrix, AUC-ROC, training time) were computed.
  - Stratified 5-fold cross-validation was performed for all three classifiers.
  - Comparison table created with correct shape and valid metric ranges.


## Optional Challenges

These challenges are clearly marked as optional and do not count toward the completion criteria.

1. **Experiment with hyperparameters:** Modify max_depth=5 for the Decision Tree, C=10 for the SVM, and hidden_layer_sizes=(50, 25) for the MLP (adding a second hidden layer). Rerun the comparison (Steps 2 and 3) and report whether adding depth to the MLP improves F1-score beyond the SVM's performance.
2. **Test on a different dataset:** Load a second binary-classification dataset from scikit-learn (e.g., using make_classification with 100 features and 500 samples) and repeat Steps 1-4 on the new data. Do the same model rankings (by F1-score) hold across different data characteristics?

## Cleanup Instructions

No cleanup is required. The notebook did not modify any files or configurations on your system. You may save the notebook to preserve your results.

## Completion Criteria

You have successfully completed this lab when:

1. All code cells run without errors and produce output.
2. Step 1 produces a train/test split with preserved class proportions (stratified split confirmed by comparing class distributions).
3. Step 2 computes and displays all six metrics (accuracy, precision, recall, F1-score, confusion matrix, AUC-ROC) and training time for all three models. All metric values are between 0 and 1.
4. Step 3 runs stratified 5-fold cross-validation and reports mean ± std accuracy for all three models across all 5 folds.
5. Step 4 produces a comparison table with 3 rows (one per model) and 9 columns (all metrics plus CV results). The table clearly shows the best performer.
6. The verification cell prints "PASS: All verification checks passed." with all sub-checks listed.
7. You have answered both comprehension questions (in a markdown cell or separate document).
8. You understand when LOOCV would be preferred over k-fold (on extremely small datasets) and why it incurs higher computational cost.